# Zadanie 1 - U-Net Segmentation

W pierwszej kolejności należy ściągnąć dane wykonując poniższe komendy w terminalu lub rozpakować je w inny sposób.

In [1]:
!wget https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz
!wget https://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz
!tar -xf images.tar.gz
!tar -xf annotations.tar.gz

--2026-11-18 04:32:21--  https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz
Resolving www.robots.ox.ac.uk (www.robots.ox.ac.uk)... 129.67.94.2
Connecting to www.robots.ox.ac.uk (www.robots.ox.ac.uk)|129.67.94.2|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://thor.robots.ox.ac.uk/pets/images.tar.gz [following]
--2026-11-18 04:32:21--  https://thor.robots.ox.ac.uk/pets/images.tar.gz
Resolving thor.robots.ox.ac.uk (thor.robots.ox.ac.uk)... 129.67.95.98
Connecting to thor.robots.ox.ac.uk (thor.robots.ox.ac.uk)|129.67.95.98|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 791918971 (755M) [application/octet-stream]
Saving to: ‘images.tar.gz’

images.tar.gz       100%[===================>] 755.23M  28.1MB/s    in 27s     

2026-11-18 04:32:48 (28.1 MB/s) - ‘images.tar.gz’ saved [791918971/791918971]

--2026-11-18 04:32:48--  https://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz
Resolving www.r

Dane przedstawiają obrazy zwierząt, łącznie z maskami segmentacji oznaczającymi lokalizację obiektu na zdjęciu z dokładnością do piksela. Zdjęcia należy wczytać jako dwa osobne datasety, jeden zawierający zdjęcia, drugą maski. Następnie należy połączyć je w jeden dataset. Należy zwrócić uwagę, aby i-te zdjęcie w pierwszej liście odpowiadało i-tej masce w drugiej liście **(shuffle=False)**

In [2]:
from pathlib import Path
import tensorflow as tf
import random

image_dir = Path("images")
mask_dir = Path("annotations/trimaps")

def names_from_split(path):
    return {
        line.split()[0]
        for line in path.read_text().splitlines()
        if line.strip() and not line.lstrip().startswith("#")
    }

images = {
    p.stem: p for p in image_dir.glob("*.jpg")
    if not p.name.startswith("._")
}
masks = {
    p.stem: p for p in mask_dir.glob("*.png")
    if not p.name.startswith("._")
}

test_names = names_from_split(Path("annotations/test.txt"))
test_stems = sorted(test_names & images.keys() & masks.keys())

missing = test_names - images.keys() | test_names - masks.keys()
if missing:
    print(f"Brak zdjęcia lub maski dla {len(missing)} nazw; przykłady: {sorted(missing)[:5]}")

def load_pair(image_path, mask_path):
    image = tf.io.read_file(image_path)
    image = tf.io.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (256, 256))
    image = image / 255.0

    mask = tf.io.read_file(mask_path)
    mask = tf.io.decode_png(mask, channels=1)
    mask = tf.image.resize(mask, (256, 256), method="nearest")
    mask = tf.cast(mask, tf.uint8) - 1

    return image, mask


trainval_names = names_from_split(Path("annotations/trainval.txt"))
trainval_stems = sorted(trainval_names & images.keys() & masks.keys())

train_dataset = (
    tf.data.Dataset.from_tensor_slices((
        [str(images[name]) for name in trainval_stems],
        [str(masks[name]) for name in trainval_stems],
    ))
    .shuffle(len(trainval_stems), reshuffle_each_iteration=True)
    .map(load_pair, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(1)
    .prefetch(tf.data.AUTOTUNE)
)

test_dataset = (
    tf.data.Dataset.from_tensor_slices((
        [str(images[name]) for name in test_stems],
        [str(masks[name]) for name in test_stems],
    ))
    .map(load_pair, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(1)
    .prefetch(tf.data.AUTOTUNE)
)

I0000 00:00:1794972774.937502   87945 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794972777.277578   87945 gpu_device.cc:2036] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4262 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2060, pci bus id: 0000:07:00.0, compute capability: 7.5


Teraz należy zaimplementować architekturę sieci. Ma ona kształt przypominający literę U a operacje zawarte w niej to konwolucje, max pooling i konwolucje transponowane. Spróbujemy oddtworzyć identyczną architekturę jak w oryginalnym artykule:

https://arxiv.org/pdf/1505.04597

Wejściem sieci jest obraz, natomiast wyjściem maska. Dobrze jest zacząć od definicji funkcji pomocniczych.

Blok konwolucji:

In [3]:
from tensorflow.keras import layers
import tensorflow as tf

def double_conv_block(x, n_filters):
    x = layers.Conv2D(
        n_filters, 3, padding="same", activation="relu",
        kernel_initializer="he_normal"
    )(x)
    x = layers.Conv2D(
        n_filters, 3, padding="same", activation="relu",
        kernel_initializer="he_normal"
    )(x)
    return x

 Blok podpróbkowania:

In [4]:
def downsample_block(x, n_filters):
    f = double_conv_block(x, n_filters)
    p = layers.MaxPool2D(pool_size=(2, 2))(f)
    p = layers.Dropout(0.3)(p)
    return f, p

 Oraz blok nadpróbkowania:

In [5]:
def upsample_block(x, conv_features, n_filters):
    x = layers.Conv2DTranspose(
        n_filters, 3, strides=2, padding="same"
    )(x)
    x = layers.Concatenate()([x, conv_features])
    x = layers.Dropout(0.3)(x)
    x = double_conv_block(x, n_filters)
    return x

 Mając do dyspozycji wszystkie operacje, można zaimplementować sieć:

In [6]:
import tensorflow as tf
def get_model(img_size, num_classes):
    inputs = layers.Input(shape=(*img_size, 3))

    f1, p1 = downsample_block(inputs, 64)
    f2, p2 = downsample_block(p1, 128)
    f3, p3 = downsample_block(p2, 256)
    f4, p4 = downsample_block(p3, 512)

    bottleneck = double_conv_block(p4, 1024)

    u6 = upsample_block(bottleneck, f4, 512)
    u7 = upsample_block(u6, f3, 256)
    u8 = upsample_block(u7, f2, 128)
    u9 = upsample_block(u8, f1, 64)

    outputs = layers.Conv2D(num_classes, 1, padding="same", activation="softmax")(u9)

    return tf.keras.Model(inputs, outputs, name="U-Net")
class SparseMeanIoU(tf.keras.metrics.MeanIoU):
    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.squeeze(tf.cast(y_true, tf.int32), axis=-1)
        y_pred = tf.argmax(y_pred, axis=-1, output_type=tf.int32)
        return super().update_state(y_true, y_pred, sample_weight)

tf.keras.backend.clear_session()

img_size = (256, 256)
num_classes = 3
model = get_model(img_size, num_classes)

Aby zweryfikować archietkurę sieci, można wypisać ją na ekran korzystając z *model.summary()*.

Przed samym rozpoczęciem uczenia, należy jeszcze zdefiniować funkcję kosztu i skompilować model. Na samym końcu wywołujemy funkcję fit z odpowiednimi argumentami.

In [7]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="pixel_accuracy"),
        SparseMeanIoU(num_classes=3, name="mean_iou"),
    ],
)

model.summary()
history = model.fit(train_dataset, epochs=20)
model.evaluate(test_dataset)

Model: "U-Net"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 256, 256,  │      1,792 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 256, 256,  │     36,928 │ conv2d[0][0]      │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 128, 128,  │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128, 128,  │          0 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 128, 128,  │     73,856 │ dropout[0][0]     │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 128, 128,  │    147,584 │ conv2d_2[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 64, 64,    │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64, 64,    │          0 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 64, 64,    │    295,168 │ dropout_1[0][0]   │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 64, 64,    │    590,080 │ conv2d_4[0][0]    │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 32, 32,    │          0 │ conv2d_5[0][0]    │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 32, 32,    │          0 │ max_pooling2d_2[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 32, 32,    │  1,180,160 │ dropout_2[0][0]   │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 32, 32,    │  2,359,808 │ conv2d_6[0][0]    │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 16, 16,    │          0 │ conv2d_7[0][0]    │
│ (MaxPooling2D)      │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 16, 16,    │          0 │ max_pooling2d_3[

 Total params: 34,513,475 (131.66 MB)

 Trainable params: 34,513,475 (131.66 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20


... (81 outputs left out)

3680/3680 ━━━━━━━━━━━━━━━━━━━━ 255s 69ms/step - loss: 0.1700 - mean_iou: 0.7940 - pixel_accuracy: 0.9321


W0000 00:00:1794977882.643959   88149 assert_op.cc:39] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
W0000 00:00:1794977882.958532   91174 hlo_rematerialization.cc:3363] Can't reduce memory use below 4.24GiB (4549153956 bytes) by rematerialization; only reduced to 4.27GiB (4587782176 bytes), down from 4.27GiB (4587782176 bytes) originally
W0000 00:00:1794977882.992644   91167 hlo_rematerialization.cc:3363] Can't reduce memory use below 4.24GiB (4549153956 bytes) by rematerialization; only reduced to 4.27GiB (4587782176 bytes), down from 4.27GiB (4587782176 bytes) originally
E0000 00:00:1794977883.890395   91164 cuda_executor.cc:1365] [0] Failed to allocate device memory of 33.30GiB (35754868736 bytes): RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
=== Source Location Trace: ===
external/xla/xla/stream_executor/cuda/cuda_status.cc:45
external/xla/xla/stream_executor/cuda/cuda_device_

3669/3669 ━━━━━━━━━━━━━━━━━━━━ 76s 20ms/step - loss: 0.3285 - mean_iou: 0.7359 - pixel_accuracy: 0.8946


[0.32845383882522583, 0.8946225047111511, 0.7359287142753601]

Skuteczność działania modelu można ocenić metryką (np. accuracy), choć przeważnie nie stosuje się jej do problemu segmentacji. W ramach ćwiczenia dodatkowego można napisać i podpiąć jedną z metryk do segmentacji (np. Dice, Jaccard). Metryka Dice nie jest dostępna w Kerasie, ze względu na wiele jej wariantów. Natomiast IoU możemy użyć z biblioteki metrics:

In [8]:
metrics=[tf.keras.metrics.MeanIoU(num_classes=3)]

# Zadanie 2 - YOLO Detection (Bonus Task)

Równie często spotykanym zadaniem w przetwarzaniu obrazów jest detekcja, polegająca na rozpozaniu prostokątów otaczających (bounding-boxes). Najczęściej stosowane są w tym celu różne warianty sieci YOLO (ang. you only look once).

Istnieje wiele gotowych, wytrenowanych wariantów sieci YOLO, które świetnie nadały by się do tego zadania. Spróbujmy jednak wytrenować własny.

W tym celu musimy jednak uzyskać etykiety. Spróbuj otrzymać je z powyższych masek segmentacji z poprzedniego zadania:


1.   Połącz klasy 1 i 2 w jedną
2.   Wyznacz prostokąt otaczający maski
3.   Zapisz koordynaty prostokąta w postaci [x, y, h, w]




In [9]:
from pathlib import Path
from PIL import Image
import numpy as np
import random
import shutil
import json

# Make an 80/20 train-validation split from the same 500-image subset
det_stems = list(trainval_stems)
random.Random(42).shuffle(det_stems)
split_at = max(1, int(0.8 * len(det_stems)))
train_stems = det_stems[:split_at]
val_stems = det_stems[split_at:]

# Build a clean YOLO-format dataset directory
yolo_root = Path("pets_yolo")
if yolo_root.exists():
    shutil.rmtree(yolo_root)

for split in ("train", "val"):
    (yolo_root / "images" / split).mkdir(parents=True, exist_ok=True)
    (yolo_root / "labels" / split).mkdir(parents=True, exist_ok=True)


def mask_to_bbox_xyhw(mask_path):
    """
    Oxford Pets trimaps: 1 = pet, 2 = background, 3 = border.
    Treat pet + border as the single foreground object.
    Return the box as (x, y, h, w), or None if no foreground pixels exist.
    """
    trimap = np.asarray(Image.open(mask_path))
    foreground = np.isin(trimap, [1, 3])

    ys, xs = np.where(foreground)
    if len(xs) == 0:
        return None

    x0, x1 = int(xs.min()), int(xs.max()) + 1
    y0, y1 = int(ys.min()), int(ys.max()) + 1
    return x0, y0, y1 - y0, x1 - x0


def write_yolo_split(stems, split):
    for stem in stems:
        image_path = images[stem]
        mask_path = masks[stem]

        with Image.open(image_path) as image:
            image_width, image_height = image.size

        with Image.open(mask_path) as mask:
            mask_width, mask_height = mask.size

        if (image_width, image_height) != (mask_width, mask_height):
            raise ValueError(f"Image/mask size mismatch for {stem}")

        box = mask_to_bbox_xyhw(mask_path)
        if box is None:
            print(f"Skipping {stem}: mask has no foreground")
            continue

        x, y, h, w = box

        # YOLO detection format: class x_center y_center width height,
        # with coordinates normalized to [0, 1].
        x_center = (x + w / 2) / image_width
        y_center = (y + h / 2) / image_height
        box_width = w / image_width
        box_height = h / image_height

        destination_image = yolo_root / "images" / split / image_path.name
        destination_label = yolo_root / "labels" / split / f"{stem}.txt"

        shutil.copy2(image_path, destination_image)
        destination_label.write_text(
            f"0 {x_center:.6f} {y_center:.6f} "
            f"{box_width:.6f} {box_height:.6f}\n"
        )


write_yolo_split(train_stems, "train")
write_yolo_split(val_stems, "val")

# YOLO dataset configuration
yaml_path = yolo_root / "data.yaml"
yaml_path.write_text(
    f"path: {json.dumps(str(yolo_root.resolve()))}\n"
    "train: images/train\n"
    "val: images/val\n"
    "names:\n"
    "  0: pet\n"
)

print(f"YOLO train images: {len(train_stems)}")
print(f"YOLO validation images: {len(val_stems)}")
print(f"Dataset config: {yaml_path}")


Skipping Egyptian_Mau_165: mask has no foreground
Skipping leonberger_18: mask has no foreground
Skipping Egyptian_Mau_162: mask has no foreground
Skipping Egyptian_Mau_196: mask has no foreground
Skipping saint_bernard_15: mask has no foreground
Skipping miniature_pinscher_14: mask has no foreground
Skipping saint_bernard_108: mask has no foreground
YOLO train images: 2944
YOLO validation images: 736
Dataset config: pets_yolo/data.yaml


Teraz pora na sieć YOLO:

4.   Zapoznaj się z działaniem architektury. Zwróć szczególną uwagę na podejścia achor-based i anchor-free.
5.   Zdecyduj czy chcesz użyć pre-trenowanej architektury (np. wytrenowanej na zbiorze COCO) i tylko ją douczyć czy może wolisz spróbować własny model
6.   Zaimplementuj architekturę i uruchom trening

In [10]:
from ultralytics import YOLO

detector = YOLO("yolo26n.pt")  # small pretrained model
detector.train(
    data=str(yaml_path),
    epochs=20,
    imgsz=640,
    batch=8,
    patience=5,
    workers=2,
    project="runs/pets",
    name="yolo_500",
    exist_ok=True,
)

# Evaluate on the 20% validation split
detector.val(data=str(yaml_path))

Ultralytics 8.4.215 🚀 Python-3.14.3 torch-2.15.0+cu132 CUDA:0 (NVIDIA GeForce RTX 2060, 5732MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=pets_yolo/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo_500, nbs=64, nms=None, opset=None, optimize=False, op

Corrupt JPEG data: premature end of data segment


       1/20      0.67G     0.9059      3.005    0.02371          7        640: 83% ━━━━━━━━━╸── 611/735 11.4it/s 58.8s<10.9s

Corrupt JPEG data: 240 extraneous bytes before marker 0xd9


       1/20      0.67G      0.916      2.721    0.02397          3        640: 100% ━━━━━━━━━━━━ 735/735 10.5it/s 1:10.1ssss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 92/92 8.6it/s 10.7s<0.1s
                   all        735        735      0.886      0.886       0.95      0.583

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       2/20     0.629G      1.011      1.287    0.02585         11        640: 8% ━─────────── 66/735 11.2it/s 5.9s<59.7s

Corrupt JPEG data: 240 extraneous bytes before marker 0xd9


... (72 outputs left out)

      20/20     0.629G     0.5594     0.4601    0.01952          4        640: 80% ━━━━━━━━━╸── 591/735 11.9it/s 50.4s<12.1s

Corrupt JPEG data: premature end of data segment


      20/20     0.629G     0.5688     0.4606    0.01987          2        640: 100% ━━━━━━━━━━━━ 735/735 11.8it/s 1:020.1sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 92/92 21.0it/s 4.4s0.1s
                   all        735        735      0.967      0.922      0.983      0.809

20 epochs completed in 0.386 hours.
Optimizer stripped from /home/user/Documents/programming/python/PracticalAspectsOfDeepLearningCourse/runs/detect/runs/pets/yolo_500/weights/last.pt, 5.4MB
Optimizer stripped from /home/user/Documents/programming/python/PracticalAspectsOfDeepLearningCourse/runs/detect/runs/pets/yolo_500/weights/best.pt, 5.4MB

Validating /home/user/Documents/programming/python/PracticalAspectsOfDeepLearningCourse/runs/detect/runs/pets/yolo_500/weights/best.pt...
Ultralytics 8.4.215 🚀 Python-3.14.3 torch-2.15.0+cu132 CUDA:0 (NVIDIA GeForce RTX 2060, 5732MiB)
YOLO26n summary (fused): 121 layers, 2,389,735 parameters, 0 gradients,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x77438c8b2e70>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

Ciekawe materiały i wizualizacje:

https://www.abhik.ai/concepts/computer-vision/anchor-based-vs-anchor-free